# small\_fable — inference only (load from Hugging Face)

Read-only. Loads the planner checkpoints trained by `train_all_colab.ipynb` **from the Hugging Face Hub**
and compares, on any prompt you give:

| model | how it answers |
|---|---|
| **Vanilla Qwen-1.5** | base `Qwen/Qwen2.5-1.5B-Instruct`, answers directly (no planner) |
| **Planned Qwen (SFT)** | emits a symbolic plan, then answers conditioned on it |
| **Planned Qwen (SFT+RL)** | same, after off-policy GRPO |

> A GPU helps but isn't required. `Runtime → Change runtime type → T4 GPU` for speed.
> The repo is **public**, so no Hugging Face token is needed to download.


## 1 · Setup — clone repo (for the model code) + install

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
%cd small_fable
!pip install -q transformers peft accelerate safetensors huggingface_hub
# Colab ships an old torchao that the installed peft rejects; we don't use it, so remove it.
!pip uninstall -y torchao >/dev/null 2>&1; echo 'torchao removed (peft/torchao version-clash workaround)'
print('ready')


## 2 · Config — which HF repo + the prompt
Set `HF_REPO` to the value the training notebook printed (`<your-username>/small_fable-planner`).
Edit `PROMPT` to whatever you want to test.

In [ ]:
HF_REPO = 'CHANGE_ME/small_fable-planner'   # <-- the repo the training notebook printed
DEVICE  = 'cuda'                            # 'cpu' also works (slower)
TEMP    = 0.7
MAX_NEW = 96

PROMPT = ('A gardener plants 3 rows of 7 tulips and 2 rows of 5 roses. '
          'If 4 tulips wilt, how many flowers are left?')
print('prompt:', PROMPT)


## 3 · Download the checkpoints from Hugging Face
Pulls `joint_ckpt/` (SFT) and `rl_ckpt/` (SFT+RL) into a local folder.

In [ ]:
from huggingface_hub import snapshot_download
local = snapshot_download(repo_id=HF_REPO, repo_type='model',
                          allow_patterns=['joint_ckpt/*', 'rl_ckpt/*'])
SFT_CKPT = os.path.join(local, 'joint_ckpt')
RL_CKPT  = os.path.join(local, 'rl_ckpt')
print('downloaded to:', local)
print('  SFT:', os.path.isdir(SFT_CKPT), '| RL:', os.path.isdir(RL_CKPT))


## 4 · Generate & compare (loads one model at a time, so it fits a T4)
Each model is loaded, run on the prompt, then freed before the next — peak memory stays ~one 1.5B
model (fp32). Vanilla Qwen via plain `transformers`; the planner models via `JointModel`.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from model_joint import JointModel, decode_plan

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'

def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

def rule(t):
    print('\n' + '=' * 70 + f'\n{t}\n' + '=' * 70)

print('PROMPT:', PROMPT)

# 1) Vanilla Qwen-1.5 — plain transformers, answers directly (no planner)
rule('1) Vanilla Qwen-1.5  (no planner)')
btok = AutoTokenizer.from_pretrained(BASE)
blm = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=(torch.float32 if DEVICE == 'cpu' else 'auto')).to(DEVICE).eval()
with torch.no_grad():
    text = btok.apply_chat_template([{'role': 'user', 'content': PROMPT}],
                                    tokenize=False, add_generation_prompt=True)
    ids = btok(text, return_tensors='pt').to(DEVICE)
    out = blm.generate(**ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP,
                       top_p=0.95, pad_token_id=btok.eos_token_id)
    print(btok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())
free(blm, btok)

# 2) & 3) Planner Qwen — emit a plan, then answer conditioned on it. One model at a time.
def show_planner(ckpt, title):
    rule(title)
    m = JointModel.from_checkpoint(BASE, ckpt, device=DEVICE); m.eval()
    with torch.no_grad():
        p_ids, p_attn = m.batch_prompts([PROMPT])
        plan = m.sample_plan(p_ids, p_attn, temp=TEMP, sample=True)
        gen = m.generate_answer(p_ids, p_attn, plan, temp=TEMP, sample=True, max_new_tokens=MAX_NEW)
        print('plan  :', ' → '.join(decode_plan(plan[0])))
        print('answer:', m.tok.decode(gen[0], skip_special_tokens=True).strip())
    free(m)

show_planner(SFT_CKPT, '2) Planned Qwen-1.5  (SFT)')
show_planner(RL_CKPT,  '3) Planned Qwen-1.5  (SFT + GRPO)')
